# 🏗️ carteira_auto — Exploração Completa do Sistema

**Objetivo:** Testar todas as funcionalidades implementadas, compreender as fontes de dados disponíveis, seus domínios e a orquestração entre os componentes.

---

### Arquitetura em Camadas

```
┌─────────────────────────────────────────────────────────────┐
│  CLI / Dashboard Streamlit                                  │
├─────────────────────────────────────────────────────────────┤
│  DAGEngine  →  Nodes (Load → Fetch → Analyze → Export)     │
├──────────┬──────────┬───────────────┬───────────────────────┤
│ Analyzers│ Alerts   │  DataLake     │  Config / Settings    │
│ (6 nodes)│ (rules+  │  (4 SQLite)   │  (dataclasses)        │
│          │  channels│               │                       │
├──────────┴──────────┼───────────────┤                       │
│  Fetchers (7)       │  Loaders /    │                       │
│  Yahoo BCB IBGE     │  Exporters    │                       │
│  FRED DDM CVM       │  (Excel)      │                       │
│  Tesouro Direto     │               │                       │
└─────────────────────┴───────────────┴───────────────────────┘
```

### Fontes de Dados por Domínio

| Domínio | Fetcher | API Key | Dados |
|---------|---------|---------|-------|
| 🇧🇷 Macro BR | BCBFetcher | Não | Selic, CDI, IPCA, PTAX, IGP-M, TR |
| 🇧🇷 Macro BR | IBGEFetcher | Não | IPCA detalhado, PIB trimestral, Desemprego |
| 🇺🇸 Macro Global | FREDFetcher | Sim (gratuita) | Fed Funds, Treasuries, VIX, CPI, BRL/USD |
| 🏛️ Renda Fixa | TesouroDiretoFetcher | Não | LFT, NTN-B, LTN, NTN-F — taxas e PU desde 2002 |
| 📈 Ações & FIIs | YahooFinanceFetcher | Não | OHLCV, fundamentos, dividendos, recomendações |
| 📋 Dados Oficiais | CVMFetcher | Não | DFP/ITR auditados, cadastro de companhias |
| 🔑 API Premium | DDMFetcher | Sim | Notícias, expectativas Focus, curvas de juros |

### Seções deste Notebook

1. **Ambiente & Config** — Setup, detecção de API keys, constantes
2. **Renda Fixa** — Tesouro Direto (curvas, taxas, histórico)
3. **Macro Brasil** — BCB + IBGE (Selic, CDI, IPCA, PIB)
4. **Macro Global** — FRED (Fed Funds, Treasuries, VIX)
5. **Ações & Fundamentos** — Yahoo Finance (preços, info, dividendos)
6. **Dados Oficiais** — CVM (DFP, mapeamento ticker→CNPJ)
7. **DataLake** — Ingestão, persistência e consulta (SQLite)
8. **Pipeline Engine** — DAG, resolução de dependências, execução
9. **Analyzers** — Portfolio, Risk, Macro, Market, Rebalancer
10. **Alertas** — Regras, canais, avaliação
11. **Resumo do Sistema** — Inventário e status

---
## 1. ⚙️ Ambiente & Configuração

In [ ]:
import os, sys, warnings, tempfile
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 100

# Detecção de API keys
from carteira_auto.config import settings

FRED_OK = bool(settings.API_KEYS.get('fred'))
DDM_OK = bool(settings.API_KEYS.get('ddm'))

print('=' * 60)
print('  carteira_auto — Ambiente de Exploração')
print('=' * 60)
print(f'  Python:       {sys.version.split()[0]}')
print(f'  Pandas:       {pd.__version__}')
print(f'  Data dir:     {settings.paths.DATA_DIR}')
print(f'  Lake dir:     {settings.paths.LAKE_DIR}')
print(f'  FRED API key: {"✅ configurada" if FRED_OK else "❌ não configurada"}')
print(f'  DDM API key:  {"✅ configurada" if DDM_OK else "❌ não configurada"}')
print('=' * 60)

---
## 2. 🏛️ Renda Fixa — Tesouro Direto

**Fonte:** Tesouro Transparente (CKAN) — dados abertos, sem autenticação
**Fetcher:** `TesouroDiretoFetcher`
**Dados:** Taxas de compra/venda e PU de todos os títulos públicos desde 2002

| Tipo | Nome Antigo | Nome Novo (2025+) | Indexador |
|------|-------------|-------------------|-----------|
| LFT | LFT | Tesouro Selic | Selic (pós-fixado) |
| NTN-B | NTN-B Principal | Tesouro IPCA+ | IPCA + spread (zero cupom) |
| NTN-B c/ cupom | NTN-B | Tesouro IPCA+ com Juros Semestrais | IPCA + spread (semestral) |
| LTN | LTN | Tesouro Prefixado | Prefixado (zero cupom) |
| NTN-F | NTN-F | Tesouro Prefixado com Juros Semestrais | Prefixado (semestral) |

In [ ]:
from carteira_auto.data.fetchers.tesouro_fetcher import TesouroDiretoFetcher

tesouro = TesouroDiretoFetcher()

# 2.1 — Taxas atuais (snapshot do último dia útil)
df_atual = tesouro.get_current_rates()
print(f'📅 Data de referência: {df_atual["data"].iloc[0].date()}')
print(f'📊 Títulos em oferta: {len(df_atual)}')
print()
df_atual[['tipo', 'vencimento', 'taxa_compra', 'taxa_venda', 'pu_compra']].sort_values('taxa_compra', ascending=False)

In [ ]:
# 2.2 — Títulos disponíveis no histórico
titulos = tesouro.get_available_titles()
print(f'📋 {len(titulos)} tipos de títulos no histórico:')
for i, t in enumerate(titulos, 1):
    print(f'   {i:2d}. {t}')

In [ ]:
# 2.3 — Curva NTN-B (IPCA+) — spread por vencimento
curve = tesouro.get_ntnb_curve()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(curve['vencimento'], curve['taxa_compra'], 'o-', color='#2196F3', markersize=8, linewidth=2)
for _, row in curve.iterrows():
    ax.annotate(f'{row["taxa_compra"]:.2f}%',
                (row['vencimento'], row['taxa_compra']),
                textcoords='offset points', xytext=(0, 12), ha='center', fontsize=9)
ax.set_title('Curva NTN-B (Tesouro IPCA+) — Taxa Real por Vencimento', fontsize=14, fontweight='bold')
ax.set_xlabel('Vencimento')
ax.set_ylabel('Taxa Compra (% a.a.)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f'Curva com {len(curve)} vértices — data base: {curve["data"].iloc[0].date()}')

In [ ]:
# 2.4 — Histórico por tipo de título (últimos 2 anos)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
tipos = [('LFT', 'Tesouro Selic', '#4CAF50'), ('NTN-B', 'Tesouro IPCA+', '#2196F3'),
         ('LTN', 'Tesouro Prefixado', '#FF9800'), ('NTN-F', 'Tesouro Prefixado c/ Juros', '#E91E63')]

for ax, (tipo, label, cor) in zip(axes.flat, tipos):
    df_tipo = tesouro.get_price_history_by_type(tipo)
    if not df_tipo.empty and 'data' in df_tipo.columns:
        # Últimos 2 anos, vencimento mais longo disponível
        recente = df_tipo[df_tipo['data'] >= pd.Timestamp.now() - pd.DateOffset(years=2)]
        if not recente.empty:
            venc_max = recente['vencimento'].max()
            serie = recente[recente['vencimento'] == venc_max].sort_values('data')
            ax.plot(serie['data'], serie['taxa_compra'], color=cor, linewidth=1.5)
            ax.set_title(f'{label} (venc. {venc_max.year})', fontweight='bold')
            ax.set_ylabel('Taxa Compra (% a.a.)')
            ax.tick_params(axis='x', rotation=30)

fig.suptitle('Evolução de Taxas — Últimos 2 Anos (vencimento mais longo)', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

---
## 3. 🇧🇷 Macro Brasil — BCB + IBGE

### BCBFetcher — Banco Central do Brasil (SGS API)
**Fonte:** Sistema Gerenciador de Séries Temporais (SGS) — sem autenticação
**Séries:** Selic meta, Selic diária, CDI, IPCA, IGP-M, PTAX, TR, PIB mensal

### IBGEFetcher — Instituto Brasileiro de Geografia e Estatística (SIDRA)
**Fonte:** API SIDRA — sem autenticação
**Séries:** IPCA detalhado (grupos), PIB trimestral, taxa de desemprego (PNAD)

In [ ]:
from carteira_auto.data.fetchers.bcb_fetcher import BCBFetcher

bcb = BCBFetcher()

# 3.1 — Indicadores BCB: panorama atual
# get_selic() retorna a Selic Meta (SGS 432) em % a.a. — frequência diária
selic = bcb.get_selic(period_days=365 * 5)
cdi = bcb.get_cdi()
ipca = bcb.get_ipca()
igpm = bcb.get_igpm()

print('=' * 55)
print('  BCB — Indicadores Macro Brasil')
print('=' * 55)
for nome, df in [('Selic Meta', selic), ('CDI', cdi), ('IPCA', ipca), ('IGP-M', igpm)]:
    ultimo = df.iloc[-1]
    print(f'  {nome:15s}: {ultimo["valor"]:8.4f}  (último: {ultimo["data"].date()})')
print('=' * 55)

In [ ]:
# 3.2 — Evolução Selic × IPCA × CDI (últimos 5 anos)
fig, ax1 = plt.subplots(figsize=(14, 6))

# Selic meta
selic_5y = selic[selic['data'] >= pd.Timestamp.now() - pd.DateOffset(years=5)]
ax1.step(selic_5y['data'], selic_5y['valor'], where='post', color='#1565C0', linewidth=2.5, label='Selic Meta')

# CDI acumulado 12m (proxy)
cdi_5y = cdi[cdi['data'] >= pd.Timestamp.now() - pd.DateOffset(years=5)]
ax1.plot(cdi_5y['data'], cdi_5y['valor'], color='#42A5F5', linewidth=1, alpha=0.6, label='CDI diário')

ax1.set_ylabel('Taxa (% a.a.)', color='#1565C0')
ax1.tick_params(axis='y', labelcolor='#1565C0')

# IPCA no eixo secundário
ax2 = ax1.twinx()
ipca_5y = ipca[ipca['data'] >= pd.Timestamp.now() - pd.DateOffset(years=5)]
ax2.bar(ipca_5y['data'], ipca_5y['valor'], width=25, alpha=0.4, color='#E53935', label='IPCA mensal')
ax2.set_ylabel('IPCA (% mês)', color='#E53935')
ax2.tick_params(axis='y', labelcolor='#E53935')

ax1.set_title('Selic Meta × CDI × IPCA — Últimos 5 Anos', fontsize=14, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
fig.tight_layout()
plt.show()

In [ ]:
# 3.3 — PTAX (câmbio BRL/USD)
ptax = bcb.get_ptax()
ptax_2y = ptax[ptax['data'] >= pd.Timestamp.now() - pd.DateOffset(years=2)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ptax_2y['data'], ptax_2y['valor'], color='#2E7D32', linewidth=1.5)
ax.fill_between(ptax_2y['data'], ptax_2y['valor'], alpha=0.1, color='#2E7D32')
ax.set_title('PTAX — Câmbio R$/USD (últimos 2 anos)', fontsize=14, fontweight='bold')
ax.set_ylabel('R$ / USD')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f'Última cotação: R$ {ptax.iloc[-1]["valor"]:.4f} em {ptax.iloc[-1]["data"].date()}')

In [ ]:
from carteira_auto.data.fetchers.ibge_fetcher import IBGEFetcher

ibge = IBGEFetcher()

# 3.4 — IBGE: IPCA detalhado por grupo + PIB + Desemprego
ipca_grupos = ibge.get_ipca_detailed()    # era: get_ipca_grupos()
pib = ibge.get_pib()                      # era: get_pib_trimestral()
desemprego = ibge.get_unemployment()      # era: get_desemprego()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# IPCA por grupo — últimos 12 meses
if not ipca_grupos.empty:
    recente = ipca_grupos.tail(12)
    if 'grupo' in recente.columns:
        pivot = recente.pivot_table(index='periodo', columns='grupo', values='valor')
        pivot.tail(6).plot(kind='bar', ax=axes[0], legend=False)
    else:
        recente.plot(x='periodo', y='valor', ax=axes[0], legend=False)
    axes[0].set_title('IPCA por Grupo', fontweight='bold')
    axes[0].set_ylabel('Variação (%)')
    axes[0].tick_params(axis='x', rotation=45)

# PIB trimestral
if not pib.empty:
    pib_plot = pib.tail(20)
    axes[1].bar(range(len(pib_plot)), pib_plot['valor'], color='#1976D2', alpha=0.8)
    axes[1].set_title('PIB Trimestral — Var. %', fontweight='bold')
    axes[1].set_ylabel('Variação (%)')

# Desemprego
if not desemprego.empty:
    desemp_plot = desemprego.tail(20)
    axes[2].plot(range(len(desemp_plot)), desemp_plot['valor'], 'o-', color='#D32F2F', linewidth=2)
    axes[2].set_title('Taxa de Desemprego (PNAD)', fontweight='bold')
    axes[2].set_ylabel('Taxa (%)')

fig.suptitle('IBGE — Indicadores Estruturais', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print(f'IPCA grupos: {len(ipca_grupos)} obs | PIB: {len(pib)} trimestres | Desemprego: {len(desemprego)} obs')

---
## 4. 🇺🇸 Macro Global — FRED

**Fonte:** Federal Reserve Economic Data (FRED) — API gratuita com chave
**Fetcher:** `FREDFetcher`
**Séries:** Fed Funds Rate, Treasury yields (2Y, 10Y, 30Y), VIX, CPI, GDP, DXY, BRL/USD

> Requer API key (`FRED_API_KEY`). Células executam apenas se a chave estiver configurada.

In [ ]:
if FRED_OK:
    from carteira_auto.data.fetchers.fred_fetcher import FREDFetcher
    fred = FREDFetcher()

    # 4.1 — Séries disponíveis no FRED
    series_map = FREDFetcher.list_series()
    print(f'📊 {len(series_map)} séries configuradas no FREDFetcher:')
    for key, info in series_map.items():
        print(f'   {key:20s} → {info["nome"]}')   # chave é "nome" (português)
else:
    print('⚠️  FRED API key não configurada — seção pulada')
    print('   Configure em .env: FRED_API_KEY=sua_chave_aqui')
    print('   Obtenha gratuitamente em: https://fred.stlouisfed.org/docs/api/api_key.html')

In [ ]:
if FRED_OK:
    # 4.2 — Painel macro EUA: Fed Funds × Treasury 10Y × VIX
    fed = fred.get_fed_funds_rate()
    t10y = fred.get_treasury_10y()
    vix = fred.get_vix()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Fed Funds
    fed_3y = fed[fed.index >= pd.Timestamp.now() - pd.DateOffset(years=3)]
    axes[0].step(fed_3y.index, fed_3y.iloc[:, 0], where='post', color='#1565C0', linewidth=2)
    axes[0].set_title('Fed Funds Rate', fontweight='bold')
    axes[0].set_ylabel('% a.a.')

    # Treasury 10Y
    t10_3y = t10y[t10y.index >= pd.Timestamp.now() - pd.DateOffset(years=3)]
    axes[1].plot(t10_3y.index, t10_3y.iloc[:, 0], color='#2E7D32', linewidth=1.5)
    axes[1].set_title('Treasury 10Y Yield', fontweight='bold')
    axes[1].set_ylabel('% a.a.')

    # VIX
    vix_1y = vix[vix.index >= pd.Timestamp.now() - pd.DateOffset(years=1)]
    axes[2].fill_between(vix_1y.index, vix_1y.iloc[:, 0], alpha=0.3, color='#E53935')
    axes[2].plot(vix_1y.index, vix_1y.iloc[:, 0], color='#E53935', linewidth=1)
    axes[2].axhline(y=20, color='gray', linestyle='--', alpha=0.5, label='VIX = 20')
    axes[2].set_title('VIX — Volatilidade Implícita', fontweight='bold')
    axes[2].legend()

    fig.suptitle('FRED — Painel Macro EUA', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()

    # Yield curve spread (10Y - 2Y)
    spread = fred.get_yield_curve_spread()
    print(f'Yield Curve Spread (10Y-2Y): {spread.iloc[-1, 0]:.2f}% — '
          f'{"Invertida ⚠️" if spread.iloc[-1, 0] < 0 else "Normal ✅"}')

---
## 5. 📈 Ações & Fundamentos — Yahoo Finance

**Fonte:** Yahoo Finance (yfinance) — sem autenticação
**Fetcher:** `YahooFinanceFetcher`
**Dados:** OHLCV diário, informações fundamentalistas, dividendos, recomendações de analistas, splits

Suporta ações brasileiras (`.SA`), ETFs, FIIs, índices globais, commodities e crypto.

In [ ]:
from carteira_auto.data.fetchers.yahoo_fetcher import YahooFinanceFetcher

yahoo = YahooFinanceFetcher()

# 5.1 — Preços históricos de ações brasileiras
tickers_br = ['PETR4.SA', 'VALE3.SA', 'ITUB4.SA', 'WEGE3.SA', 'ABEV3.SA']
df_precos = yahoo.get_historical_price_data(tickers_br, period='1y')

fig, ax = plt.subplots(figsize=(14, 6))
for ticker in tickers_br:
    col = [c for c in df_precos.columns if ticker.replace('.SA', '') in str(c) and 'Close' in str(c)]
    if col:
        serie = df_precos[col[0]].dropna()
        # Normaliza para base 100
        serie_norm = (serie / serie.iloc[0]) * 100
        ax.plot(serie_norm.index, serie_norm, linewidth=1.5, label=ticker.replace('.SA', ''))

ax.axhline(y=100, color='gray', linestyle='--', alpha=0.3)
ax.set_title('Ações Brasileiras — Performance Relativa (base 100, 1 ano)', fontsize=14, fontweight='bold')
ax.set_ylabel('Base 100')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# 5.2 — Informações fundamentalistas
info = yahoo.get_basic_info('PETR4.SA')
if info:
    campos = ['shortName', 'sector', 'industry', 'marketCap', 'trailingPE',
              'forwardPE', 'dividendYield', 'beta', 'fiftyTwoWeekHigh', 'fiftyTwoWeekLow']
    print('📋 PETR4 — Dados Fundamentalistas')
    print('-' * 45)
    for c in campos:
        val = info.get(c, 'N/D')
        if isinstance(val, float):
            if 'Yield' in c:
                val = f'{val*100:.2f}%'
            elif 'Cap' in c:
                val = f'R$ {val/1e9:.1f}B'
            else:
                val = f'{val:.2f}'
        print(f'  {c:22s}: {val}')

In [ ]:
# 5.3 — Dividendos e recomendações
divs = yahoo.get_dividends('PETR4.SA')
recs = yahoo.get_recommendations('PETR4.SA')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histórico de dividendos
if divs and 'dividends' in divs:
    df_div = divs['dividends']
    if isinstance(df_div, pd.DataFrame) and not df_div.empty:
        recent_div = df_div.tail(20)
        axes[0].bar(range(len(recent_div)), recent_div.iloc[:, 0], color='#4CAF50', alpha=0.8)
        axes[0].set_title('PETR4 — Últimos 20 Dividendos', fontweight='bold')
        axes[0].set_ylabel('R$ / ação')
    elif isinstance(df_div, pd.Series) and not df_div.empty:
        recent_div = df_div.tail(20)
        axes[0].bar(range(len(recent_div)), recent_div.values, color='#4CAF50', alpha=0.8)
        axes[0].set_title('PETR4 — Últimos 20 Dividendos', fontweight='bold')
        axes[0].set_ylabel('R$ / ação')

# Recomendações de analistas
if recs and isinstance(recs, dict):
    rec_data = recs.get('recommendations', recs)
    if isinstance(rec_data, pd.DataFrame) and not rec_data.empty:
        axes[1].set_title('PETR4 — Recomendações', fontweight='bold')
        rec_data.tail(5).plot(kind='barh', ax=axes[1])
    else:
        axes[1].text(0.5, 0.5, 'Dados indisponíveis', ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('PETR4 — Recomendações', fontweight='bold')

fig.tight_layout()
plt.show()

# Batch info — múltiplos ativos
batch = yahoo.get_batch_info(['PETR4.SA', 'VALE3.SA', 'WEGE3.SA'], fields=['shortName', 'trailingPE', 'dividendYield'])
print('\n📊 Comparação rápida:')
for ticker, data in batch.items():
    pe = data.get('trailingPE', 'N/D')
    dy = data.get('dividendYield', 0) or 0
    print(f'  {ticker:10s}: P/L = {pe if isinstance(pe, str) else f"{pe:.1f}":>6s}  |  DY = {dy*100:.1f}%')

In [ ]:
# 5.4 — Commodities, índices globais e crypto (via Yahoo)
global_tickers = {
    'Petróleo (WTI)': 'CL=F',
    'Ouro': 'GC=F',
    'S&P 500': '^GSPC',
    'IBOVESPA': '^BVSP',
    'Bitcoin': 'BTC-USD',
}

precos_global = {}
for nome, ticker in global_tickers.items():
    p = yahoo.get_current_price(ticker)
    precos_global[nome] = p

print('🌍 Preços Globais (Yahoo Finance)')
print('-' * 40)
for nome, preco in precos_global.items():
    if preco:
        print(f'  {nome:18s}: {preco:>12,.2f}')
    else:
        print(f'  {nome:18s}: indisponível')

---
## 6. 📋 Dados Oficiais — CVM

**Fonte:** Comissão de Valores Mobiliários — dados abertos (sem autenticação)
**Fetcher:** `CVMFetcher`
**Dados:** Cadastro de companhias abertas, DFP/ITR (demonstrações financeiras auditadas), mapeamento ticker→CNPJ

O mapeamento ticker→CNPJ usa heurística de prefixo sobre `denom_comerc` (nome comercial) do cadastro CVM. Funciona bem para empresas cujo ticker reflete o nome (PETR4→PETROBRAS, VALE3→VALE), mas pode falhar para abreviações não-óbvias (ITUB4→Itaú Unibanco).

In [ ]:
from carteira_auto.data.fetchers.cvm_fetcher import CVMFetcher

cvm = CVMFetcher()

# 6.1 — Cadastro de companhias abertas
registry = cvm.get_company_registry()
print(f'📋 Cadastro CVM: {len(registry)} companhias')
print(f'   Colunas: {list(registry.columns)}')
print()

# Amostra de companhias ativas
if 'situacao' in registry.columns:
    ativas = registry[registry['situacao'].str.upper().str.startswith('A', na=False)]
    print(f'   Ativas: {len(ativas)}')
    cols_show = [c for c in ['razao_social', 'nome_pregao', 'cnpj', 'setor'] if c in ativas.columns]
    display(ativas[cols_show].head(10))

In [ ]:
# 6.2 — Mapeamento ticker → CNPJ
tickers_teste = ['PETR4', 'VALE3', 'WEGE3', 'EMBR3', 'ITUB4', 'ABEV3', 'BBDC4', 'BBAS3']

print('🔗 Mapeamento Ticker → CNPJ (heurística de prefixo sobre denom_comerc)')
print('-' * 65)
for ticker in tickers_teste:
    cnpj = cvm.get_cnpj_by_ticker(ticker)
    status = f'✅ {cnpj}' if cnpj else '❌ não encontrado'
    print(f'  {ticker:8s} → {status}')

In [ ]:
# 6.3 — DFP (Demonstrações Financeiras Padronizadas) — via ticker
# Tenta buscar DRE de PETR4 do último exercício
try:
    dre = cvm.get_dfp_by_ticker('PETR4', year=2024, statement='DRE')
    if not dre.empty:
        print(f'📊 DRE PETR4 (DFP 2024): {len(dre)} linhas')
        display(dre.head(15))
    else:
        print('⚠️  DFP 2024 ainda não disponível — tentando 2023...')
        dre = cvm.get_dfp_by_ticker('PETR4', year=2023, statement='DRE')
        if not dre.empty:
            print(f'📊 DRE PETR4 (DFP 2023): {len(dre)} linhas')
            display(dre.head(15))
        else:
            print('⚠️  DFP indisponível para PETR4')
except Exception as e:
    print(f'⚠️  Erro ao buscar DFP: {e}')

---
## 7. 💾 DataLake — Ingestão, Persistência e Consulta

**Backend:** SQLite (4 databases especializados)
**Classe unificada:** `DataLake` (fachada para PriceLake, MacroLake, FundamentalsLake, NewsLake)

| Lake | Arquivo SQLite | Dados | Exporta Parquet |
|------|---------------|-------|----------------|
| PriceLake | `prices.db` | OHLCV por ticker/data | Sim |
| MacroLake | `macro.db` | Séries macro (BCB, IBGE, FRED) | Sim |
| FundamentalsLake | `fundamentals.db` | DRE, Balanço, indicadores | Sim |
| NewsLake | `news.db` | Notícias + sentimento | Sim |

Fluxo: **Fetcher → DataLake.store_*() → SQLite → DataLake.get_*() → Analyzer**

In [ ]:
from carteira_auto.data.lake import DataLake

# 7.1 — Inicialização e sumário do DataLake
lake = DataLake(lake_dir=settings.paths.LAKE_DIR)

summary = lake.summary()
print('💾 DataLake — Estado Atual')
print('=' * 50)
for key, val in summary.items():
    print(f'  {key:25s}: {val}')
print('=' * 50)

In [ ]:
# 7.2 — Ingestão de preços no PriceLake (demo com dados do Yahoo)
demo_tickers = ['PETR4.SA', 'VALE3.SA']
df_demo = yahoo.get_historical_price_data(demo_tickers, period='6mo')

if not df_demo.empty:
    n_stored = lake.store_prices(df_demo, source='yahoo')
    print(f'✅ {n_stored} registros de preço armazenados no PriceLake')

    # Consulta de volta
    df_back = lake.get_prices(['PETR4.SA', 'VALE3.SA'])
    print(f'📊 Consulta: {len(df_back)} registros retornados')

    # Preço mais recente
    latest = lake.get_latest_prices(['PETR4.SA', 'VALE3.SA'])
    for ticker, price in latest.items():
        print(f'   {ticker}: R$ {price:.2f}')
else:
    print('⚠️  Sem dados de preço para armazenar')

In [ ]:
# 7.3 — Ingestão de macro no MacroLake (demo com Selic do BCB)
n_macro = lake.store_macro(
    indicator='selic_meta',
    df=selic.rename(columns={'data': 'date', 'valor': 'value'}).set_index('date'),
    source='bcb',
    unit='% a.a.',
    frequency='daily'
)
print(f'✅ {n_macro} registros de Selic Meta armazenados no MacroLake')

# Consulta
df_selic_lake = lake.get_macro('selic_meta')
print(f'📊 Consulta MacroLake: {len(df_selic_lake)} registros')

# Último valor
ultimo = lake.get_macro_latest('selic_meta')
print(f'   Último valor: {ultimo}')

In [ ]:
# 7.4 — Exportação para Parquet (para ML / análise offline)
with tempfile.TemporaryDirectory() as tmpdir:
    exports = lake.export_all_to_parquet()
    print('📦 Exportação Parquet:')
    for name, path in exports.items():
        if path and path.exists():
            size_kb = path.stat().st_size / 1024
            print(f'   {name:20s} → {path.name} ({size_kb:.1f} KB)')
        else:
            print(f'   {name:20s} → (vazio ou inexistente)')

# Sumário atualizado
summary = lake.summary()
print(f'\n💾 Sumário atualizado: {summary}')

---
## 8. ⚙️ Pipeline Engine — DAG, Nodes e Orquestração

**Core:** `DAGEngine` com resolução topológica (Kahn's Algorithm)
**Abstração:** `Node` ABC → `name`, `dependencies`, `run(ctx)`
**Comunicação:** `PipelineContext` (dicionário tipado entre Nodes)

### Pipelines Registradas

| Pipeline | Descrição | Nodes |
|----------|-----------|-------|
| `update-excel-portfolio-prices` | Atualiza planilha com preços | Load → Fetch → Export |
| `analyze` | Análise completa do portfólio | Load → Fetch → Portfolio → Risk → Market → Macro |
| `rebalance` | Sugestão de rebalanceamento | Load → Fetch → Portfolio → Rebalancer |
| `ingest-prices` | Ingestão de preços no lake | IngestPrices |
| `ingest-macro` | Ingestão de dados macro | IngestMacro |
| `ingest-fundamentals` | Ingestão de fundamentos | IngestFundamentals |
| `ingest-news` | Ingestão de notícias | IngestNews |

In [ ]:
from carteira_auto.core.engine import DAGEngine, Node, PipelineContext
from carteira_auto.core.registry import PIPELINE_PRESETS, create_engine

# 8.1 — Pipelines disponíveis e seus nodes
print('🔧 Pipelines Registradas (PIPELINE_PRESETS)')
print('=' * 60)
for name, config in PIPELINE_PRESETS.items():
    target = config.get('target', '?')
    nodes = config.get('nodes', [])
    node_names = [n.name if hasattr(n, 'name') else type(n).__name__ for n in nodes]
    print(f'  {name:35s} → target: {target}')
    print(f'    {"nodes:":35s}   {" → ".join(node_names)}')
print('=' * 60)

In [ ]:
# 8.2 — Resolução de dependências (dry run)
engine = create_engine()

print('🔍 Resolução topológica — dry_run por pipeline:')
print('-' * 50)
for name, config in PIPELINE_PRESETS.items():
    target = config.get('target', None)
    if target:
        try:
            order = engine.dry_run(target)
            print(f'  {name}: {" → ".join(order)}')
        except Exception as e:
            print(f'  {name}: ⚠️ {e}')

print(f'\n📋 Total de nodes registrados: {len(engine.list_nodes())}')
print(f'   Nodes: {", ".join(engine.list_nodes())}')

In [ ]:
# 8.3 — Execução real: pipeline de ingestão de preços
print('▶️  Executando pipeline: ingest-prices')
print('-' * 50)
try:
    ctx = engine.run('ingest_prices')
    print('✅ Pipeline ingest-prices concluída com sucesso')
    # Mostrar chaves do contexto resultante
    print(f'   Chaves no contexto: {list(ctx.keys()) if isinstance(ctx, dict) else "PipelineContext"}')
except Exception as e:
    print(f'⚠️  Erro na pipeline: {e}')

---
## 9. 📊 Analyzers — Análise de Portfólio, Risco e Macro

Cada analyzer é um `Node` que recebe e produz dados via `PipelineContext`. Podem ser executados individualmente ou orquestrados pelo `DAGEngine`.

| Analyzer | Responsabilidade | Produz |
|----------|-----------------|--------|
| `PortfolioAnalyzer` | Alocação, retorno, diversificação | `portfolio_metrics` |
| `RiskAnalyzer` | Volatilidade, VaR, Sharpe, drawdown | `risk_metrics` |
| `MacroAnalyzer` | Selic, IPCA, cenário macro BR | `macro_context` |
| `MarketAnalyzer` | Tendências, correlações de mercado | `market_metrics` |
| `MarketSectorAnalyzer` | Performance por setor de mercado | `sector_metrics` |
| `EconomicSectorAnalyzer` | Análise setorial econômica | `economic_sector_metrics` |
| `Rebalancer` | Sugestão de rebalanceamento | `rebalance_recommendations` |

In [ ]:
# 9.1 — Executar pipeline de análise completa
# Requer a planilha de portfólio configurada em settings.paths
from carteira_auto.analyzers import (
    PortfolioAnalyzer, RiskAnalyzer, MacroAnalyzer,
    MarketAnalyzer, Rebalancer
)

# Inventário dos analyzers disponíveis
analyzers = [PortfolioAnalyzer, RiskAnalyzer, MacroAnalyzer, MarketAnalyzer, Rebalancer]

print('📊 Analyzers Disponíveis')
print('=' * 60)
for cls in analyzers:
    instance = cls()
    print(f'  {instance.name:30s} deps: {instance.dependencies}')
print('=' * 60)

# Tentar executar pipeline analyze
try:
    engine_analyze = create_engine()
    ctx = engine_analyze.run('analyze')
    print('\n✅ Pipeline "analyze" executada com sucesso!')
    
    # Mostrar métricas do portfólio se disponíveis
    if 'portfolio_metrics' in ctx:
        pm = ctx['portfolio_metrics']
        print(f'\n📋 Métricas do Portfólio:')
        if hasattr(pm, '__dict__'):
            for k, v in vars(pm).items():
                if not k.startswith('_'):
                    print(f'   {k}: {v}')
        elif isinstance(pm, dict):
            for k, v in pm.items():
                print(f'   {k}: {v}')
    
    # Mostrar métricas de risco se disponíveis
    if 'risk_metrics' in ctx:
        rm = ctx['risk_metrics']
        print(f'\n📋 Métricas de Risco:')
        if hasattr(rm, '__dict__'):
            for k, v in vars(rm).items():
                if not k.startswith('_'):
                    print(f'   {k}: {v}')
        elif isinstance(rm, dict):
            for k, v in rm.items():
                print(f'   {k}: {v}')
                
except FileNotFoundError:
    print('\n⚠️  Planilha de portfólio não encontrada.')
    print('   Configure o caminho em settings.paths.SOURCE_FILE')
    print('   A pipeline "analyze" requer LoadPortfolioNode → FetchPricesNode → Analyzers')
except Exception as e:
    print(f'\n⚠️  Erro: {e}')

---
## 10. 🔔 Alertas — Regras, Canais e Avaliação

**Componentes:**
- `AlertRule` — Regra com condição, threshold e severidade
- `AlertEngine` — Avalia regras contra um contexto
- `AlertChannel` (ABC) → `ConsoleChannel`, `LogChannel` (extensível para Telegram, Email)
- `rules.py` — Regras predefinidas: rebalance_alert, price_drop_alert, selic_change_alert

In [ ]:
from carteira_auto.alerts import AlertEngine, AlertRule, ConsoleChannel, LogChannel
from carteira_auto.alerts.rules import rebalance_alert, price_drop_alert, selic_change_alert

# 10.1 — Regras predefinidas
regras = [
    rebalance_alert(threshold=0.05),
    price_drop_alert(threshold=0.10),
    selic_change_alert(threshold=0.25),
]

print('📋 Regras de Alerta Predefinidas')
print('=' * 65)
for r in regras:
    print(f'  {r.name:25s} | cond: {r.condition:20s} | threshold: {r.threshold} | sev: {r.severity}')
print('=' * 65)

# 10.2 — Avaliação de regras (simulação)
alert_engine = AlertEngine()
alert_engine.register_many(regras)

# Contexto simulado com desvio de alocação
ctx_simulado = {
    'max_deviation': 0.08,    # 8% de desvio → aciona rebalance (threshold 5%)
    'max_price_drop': 0.05,   # 5% de queda → NÃO aciona price_drop (threshold 10%)
    'selic_change': 0.50,     # 0.50pp de mudança → aciona selic_change (threshold 0.25)
}

alerts = alert_engine.evaluate(ctx_simulado)
print(f'\n🔔 Alertas disparados: {len(alerts)}')
for a in alerts:
    print(f'  ⚠️  [{a.rule.severity.upper()}] {a.rule.name}: {a.message}')

if not alerts:
    print('  ✅ Nenhum alerta disparado — portfólio dentro dos limites')

# 10.3 — Canais de notificação
console = ConsoleChannel()
for a in alerts:
    console.send(a)

---
## 11. 📋 Resumo do Sistema — Inventário Completo

### Visão geral da arquitetura e componentes implementados

In [ ]:
# 11.1 — Inventário completo do sistema
import importlib
import pkgutil

def count_classes(module_path, base_class=None):
    """Conta classes em um pacote."""
    import inspect
    count = 0
    try:
        mod = importlib.import_module(module_path)
        for name, obj in inspect.getmembers(mod, inspect.isclass):
            if base_class is None or (hasattr(obj, '__mro__') and base_class in [c.__name__ for c in obj.__mro__]):
                count += 1
    except Exception:
        pass
    return count

print('╔══════════════════════════════════════════════════════════════╗')
print('║            carteira_auto — Inventário do Sistema            ║')
print('╠══════════════════════════════════════════════════════════════╣')

# Fetchers
fetchers = {
    'YahooFinanceFetcher': ('Ações, FIIs, ETFs, índices, commodities, crypto', '❌ Não'),
    'BCBFetcher': ('Selic, CDI, IPCA, PTAX, IGP-M, TR', '❌ Não'),
    'IBGEFetcher': ('IPCA grupos, PIB trimestral, Desemprego', '❌ Não'),
    'FREDFetcher': ('Fed Funds, Treasuries, VIX, CPI, GDP', '✅ FRED_API_KEY'),
    'TesouroDiretoFetcher': ('LFT, NTN-B, LTN, NTN-F — taxas e PU', '❌ Não'),
    'CVMFetcher': ('Cadastro, DFP, ITR, ticker→CNPJ', '❌ Não'),
    'DDMFetcher': ('Notícias, Focus, curvas de juros', '✅ DDM_API_KEY'),
}

print('║                                                              ║')
print('║  📡 FETCHERS (7)                                             ║')
print('║  ──────────────────────────────────────────────────────────  ║')
for name, (dados, key) in fetchers.items():
    print(f'║  {name:25s} {key:15s} {dados[:35]:35s} ║')

# DataLake
print('║                                                              ║')
print('║  💾 DATALAKE (4 SQLite)                                      ║')
print('║  ──────────────────────────────────────────────────────────  ║')
lakes = ['PriceLake (prices.db)', 'MacroLake (macro.db)',
         'FundamentalsLake (fundamentals.db)', 'NewsLake (news.db)']
for l in lakes:
    print(f'║    {l:56s}  ║')

# Analyzers
print('║                                                              ║')
print('║  📊 ANALYZERS (7)                                            ║')
print('║  ──────────────────────────────────────────────────────────  ║')
analyzer_list = ['PortfolioAnalyzer', 'RiskAnalyzer', 'MacroAnalyzer',
                 'MarketAnalyzer', 'MarketSectorAnalyzer',
                 'EconomicSectorAnalyzer', 'Rebalancer']
for a in analyzer_list:
    print(f'║    {a:56s}  ║')

# Pipelines
print('║                                                              ║')
print('║  ⚙️  PIPELINES ({:d})                                         ║'.format(len(PIPELINE_PRESETS)))
print('║  ──────────────────────────────────────────────────────────  ║')
for p in PIPELINE_PRESETS:
    print(f'║    {p:56s}  ║')

print('║                                                              ║')
print('╚══════════════════════════════════════════════════════════════╝')

# Status das API keys
print(f'\n🔑 FRED: {"✅" if FRED_OK else "❌"}  |  DDM: {"✅" if DDM_OK else "❌"}')
print(f'💾 Lake dir: {settings.paths.LAKE_DIR}')
print(f'📊 Lake summary: {lake.summary()}')

---

### Próximos Passos (Fases 2-7)

| Fase | Escopo | Status |
|------|--------|--------|
| 0 | DataLake + IngestNodes + Configs | ✅ Implementado |
| 1 | FRED + CVM + Tesouro Direto + Enrichment | ✅ Implementado |
| 2 | 11 Analyzers avançados (fundamental, currency, commodity...) | 🔲 Planejado |
| 3 | Estratégias + Optimizer (PyPortfolioOpt) + Backtesting | 🔲 Planejado |
| 4 | ML Pipeline: scoring fundamentalista, SHAP | 🔲 Planejado |
| 5 | NLP: sentimento (FinBERT), geopolítica, crisis hedge | 🔲 Planejado |
| 6 | AI Reasoning: Claude/Deepseek, PromptEngine | 🔲 Planejado |
| 7 | Publishers: Telegram, PDF, Email, Scheduler | 🔲 Planejado |

---
*Notebook gerado automaticamente para exploração do sistema carteira_auto v0.1.0*

---
## 4.5 🔑 API Premium — DDM (Dados de Mercado)

**Fonte:** DadosDeMercado.com.br — API premium com chave
**Fetcher:** `DDMFetcher`
**Dados:** 35 endpoints cobrindo ações, FIIs, fundos, Tesouro, macro, câmbio e notícias

| Grupo | Endpoints chave |
|-------|----------------|
| Ações | `get_companies`, `get_stock_data`, `get_financials`, `get_quotations` |
| FIIs/Fundos | `get_fii_list`, `get_fii_dividends`, `get_fund_list` |
| Tesouro | `get_treasury_list`, `get_treasury_prices`, `get_treasury_price_history` |
| Macro | `get_economic_indices`, `get_focus_bulletin`, `get_interest_curves` |
| Câmbio | `get_currencies`, `get_currency_conversion` |
| Notícias | `get_news(ticker)` |

> Requer `DDM_API_KEY`. Obtenha em dadosdemercado.com.br.

In [ ]:
if DDM_OK:
    from carteira_auto.data.fetchers.ddm_fetcher import DDMFetcher
    ddm = DDMFetcher()

    # Status atual da API — mapear o que está funcional
    print('🔍 Verificando disponibilidade dos endpoints DDM...')
    endpoints_status = {}
    test_methods = {
        'get_economic_indices': 'Macro — índices econômicos',
        'get_focus_bulletin': 'Macro — boletim Focus',
        'get_interest_curves': 'Macro — curvas de juros',
        'get_companies': 'Ações — cadastro de empresas',
        'get_treasury_list': 'Tesouro — lista',
        'get_fii_list': 'FIIs — lista',
        'get_currencies': 'Câmbio — moedas',
        'get_risk_indicators': 'Risco — indicadores',
    }
    for method, desc in test_methods.items():
        try:
            result = getattr(ddm, method)()
            endpoints_status[method] = ('✅', len(result))
        except Exception as e:
            code = str(e).split()[0] if str(e) else '?'
            endpoints_status[method] = ('❌', str(e)[:60])

    print(f'\n{"Endpoint":<30} {"Status":<5} {"Detalhe"}')
    print('-' * 75)
    for method, (status, detail) in endpoints_status.items():
        desc = test_methods[method]
        print(f'  {desc:<30} {status}  {detail}')

    # Executar endpoints que estejam funcionais
    funcionais = [m for m, (s, _) in endpoints_status.items() if s == '✅']
    if funcionais:
        print(f'\n✅ {len(funcionais)} endpoints funcionais — explorando...')
        for method in funcionais[:3]:
            result = getattr(ddm, method)()
            print(f'\n{method}(): {len(result)} itens')
            if result and isinstance(result[0], dict):
                print(f'  Campos: {list(result[0].keys())[:8]}')
    else:
        print('\n⚠️  Nenhum endpoint DDM disponível no momento.')
        print('   Possíveis causas: migração de API, plano expirado ou manutenção.')
        print('   O DDMFetcher está implementado e funcional — requer apenas API ativa.')

else:
    print('⚠️  DDM API key não configurada — seção pulada')
    print('   Configure em .env: DDM_API_KEY=sua_chave_aqui')
    print()
    print('📋 Métodos disponíveis no DDMFetcher (35 endpoints):')
    grupos = {
        'Ações': ['get_companies', 'get_stock_data', 'get_financials', 'get_quotations', 'get_dividends'],
        'FIIs/Fundos': ['get_fii_list', 'get_fii_dividends', 'get_fund_list'],
        'Tesouro': ['get_treasury_list', 'get_treasury_prices', 'get_treasury_price_history'],
        'Macro': ['get_economic_indices', 'get_focus_bulletin', 'get_interest_curves', 'get_market_expectations'],
        'Câmbio': ['get_currencies', 'get_currency_conversion'],
        'Notícias': ['get_news'],
    }
    for grupo, metodos in grupos.items():
        print(f'  {grupo}: {", ".join(metodos)}')